In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
import os
import imageio
import pandas as pd
import json


In [ ]:
file_path = "data_cleaned.csv"


In [ ]:
data_cleaned = pd.read_csv(file_path)
df = data_cleaned.copy()


In [ ]:
df.head()

In [ ]:
raw_date = df["date"].astype(str).str.strip()

# Primary format in current cleaned file is mm/dd/yy (e.g., 3/5/20).
parsed_date = pd.to_datetime(raw_date, format="%m/%d/%y", errors="coerce")

# Fallback for potential mm/dd/YYYY rows.
fallback_mask = parsed_date.isna()
if fallback_mask.any():
    parsed_date.loc[fallback_mask] = pd.to_datetime(
        raw_date.loc[fallback_mask],
        format="%m/%d/%Y",
        errors="coerce"
    )

# Final generic fallback for any remaining irregular cases.
fallback_mask = parsed_date.isna()
if fallback_mask.any():
    parsed_date.loc[fallback_mask] = pd.to_datetime(raw_date.loc[fallback_mask], errors="coerce")

df["date"] = parsed_date
print("Missing parsed dates:", int(df["date"].isna().sum()))
print("Date range:", df["date"].min(), "to", df["date"].max())


id                               object
source                           object
date                     datetime64[ns]
broad_topic                      object
topic_summary                    object
sentiment_polarity              float64
emotion                          object
narrative_frame                  object
value_dimension                  object
value_stance                     object
confidence                      float64
broad_topic_clean                object
narrative_frame_clean            object
emotion_clean                    object
value_dimension_clean            object
value_stance_clean               object
general_topic                    object
general_narrative                object
dtype: object


In [ ]:
df[df["date"].isna()]

,id,source,date,broad_topic,topic_summary,sentiment_polarity,emotion,narrative_frame,value_dimension,value_stance,confidence,broad_topic_clean,narrative_frame_clean,emotion_clean,value_dimension_clean,value_stance_clean,general_topic,general_narrative


In [ ]:
# Stable sort preserves within-day order while enforcing chronology for tick updates.
df = df.sort_values(by="date", kind="mergesort").reset_index(drop=True)
df.head()


,id,source,date,broad_topic,topic_summary,sentiment_polarity,emotion,narrative_frame,value_dimension,value_stance,confidence,broad_topic_clean,narrative_frame_clean,emotion_clean,value_dimension_clean,value_stance_clean,general_topic,general_narrative
0,reddit_euviut,reddit,2020-01-27,Conspiracy Theories & Misinformation,coronavirus misinformation and fake news,-0.50,outrage,misinformation and conspiracy narratives,science_conspiracy,conspiracy,0.9,Conspiracy Theories & Misinformation,misinformation and conspiracy narratives,outrage,science_conspiracy,conspiracy,Conspiracy Theories & Misinformation,Misinformation & Conspiracy Narratives
1,reddit_evvup1,reddit,2020-01-29,"Virus Origins, Lab Theories & Biology",coronavirus spread predictions,-0.55,outrage,threat amplification / media-driven fear,optimism_pessimism,pessimism,0.8,"Virus Origins, Lab Theories & Biology",threat amplification / media-driven fear,outrage,optimism_pessimism,pessimism,"Virus Origins, Lab Theories & Biology",Threat Amplification & Fear
2,reddit_ew111p,reddit,2020-01-30,Conspiracy Theories & Misinformation,coronavirus fatality prediction model skepticism,-0.60,outrage,threat amplification / media-driven fear,trust_distrust,distrust,0.9,Conspiracy Theories & Misinformation,threat amplification / media-driven fear,outrage,trust_distrust,distrust,Conspiracy Theories & Misinformation,Threat Amplification & Fear
3,reddit_ewfast,reddit,2020-01-31,"Virus Origins, Lab Theories & Biology",coronavirus origins video warning,-0.50,fear,threat amplification / media-driven fear,safety_threat,threat,0.9,"Virus Origins, Lab Theories & Biology",threat amplification / media-driven fear,fear,safety_threat,threat,"Virus Origins, Lab Theories & Biology",Threat Amplification & Fear
4,news_114,news,2020-02-07,Social & Economic Impact,Medical mask shortage due to China's outbreak ...,-0.50,anxiety,threat amplification / media-driven fear,safety_threat,threat,0.8,Social & Economic Impact,threat amplification / media-driven fear,anxiety,safety_threat,threat,Social & Economic Impact,Threat Amplification & Fear


## Data Processing to Construct Topic-Narrative Imprint Matrix


#### Step 1: Define cleaned topic/narrative mappings


In [ ]:
preferred_topic_order = [
    "virus origins, lab theories & biology",
    "treatment, symptoms & hospitals",
    "travel, mobility & border control",
    "lockdowns, mandates & social distancing",
    "vaccines & immunity",
    "politics, government & policy",
    "social & economic impact",
    "conspiracy theories & misinformation",
    "general/other",
]

preferred_narrative_order = [
    "government transparency & trust",
    "institutional competence & failure",
    "threat amplification & fear",
    "misinformation & conspiracy narratives",
    "public health guidance & compliance",
    "scientific evidence & uncertainty",
    "solidarity & mutual protection",
    "general/other",
]


#### Step 2: Map cleaned dataframe categories to indices


In [ ]:
topic_col = "general_topic"
narrative_col = "general_narrative"

required_cols = [topic_col, narrative_col, "sentiment_polarity", "source", "date"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

df[topic_col] = df[topic_col].astype(str).str.strip().str.lower()
df[narrative_col] = df[narrative_col].astype(str).str.strip().str.lower()

observed_topics = set(df[topic_col].dropna().tolist())
observed_narratives = set(df[narrative_col].dropna().tolist())

topic_order = [t for t in preferred_topic_order if t in observed_topics]
topic_order += sorted(observed_topics - set(topic_order))

narrative_order = [n for n in preferred_narrative_order if n in observed_narratives]
narrative_order += sorted(observed_narratives - set(narrative_order))

topic_to_row = {t: i for i, t in enumerate(topic_order)}
narrative_to_col = {n: j for j, n in enumerate(narrative_order)}

df["row_idx"] = df[topic_col].map(topic_to_row)
df["col_idx"] = df[narrative_col].map(narrative_to_col)

missing_rows = int(df["row_idx"].isna().sum())
missing_cols = int(df["col_idx"].isna().sum())
print(f"Topic categories: {len(topic_order)}, Narrative categories: {len(narrative_order)}")
print(f"Missing topic mappings: {missing_rows}, Missing narrative mappings: {missing_cols}")


In [ ]:
df[[topic_col, "row_idx"]].head()


,emotion_clean,row_idx
0,outrage,5
1,outrage,5
2,outrage,5
3,fear,2
4,anxiety,1


In [ ]:
df[[narrative_col, "col_idx"]].head()


,narrative_frame_clean,col_idx
0,misinformation and conspiracy narratives,3
1,threat amplification / media-driven fear,7
2,threat amplification / media-driven fear,7
3,threat amplification / media-driven fear,7
4,threat amplification / media-driven fear,7


#### Step 3a: Construct Topic-Narrative Imprint Matrix and Message Dictionary


In [ ]:
eim_matrix_shape = (len(topic_order), len(narrative_order))
message_passing_dictionary = {}

tick = 0
for _, r in df.iterrows():
    if pd.isna(r["row_idx"]) or pd.isna(r["col_idx"]):
        continue

    tick += 1
    eim_matrix = np.zeros(eim_matrix_shape)

    i = int(r["row_idx"])
    j = int(r["col_idx"])
    polarity = float(r["sentiment_polarity"])

    eim_matrix[i, j] = polarity

    information_source = r["source"]
    label = True
    key = (tick, information_source, label)
    message_passing_dictionary[key] = eim_matrix

print(f"EIM shape (topic x narrative): {eim_matrix_shape}")
print(f"Messages encoded into EIM dictionary: {len(message_passing_dictionary)}")


#### Step 3b: Make sure Message Passing Dictionary was Constructed Correctly

In [ ]:
for k in list(message_passing_dictionary.keys())[:5]:
    print(k)
    print(message_passing_dictionary[k])

(1, 'reddit', True)
[[ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.  -0.5  0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]]
(2, 'reddit', True)
[[ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.   -0.55  0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.  ]]
(3, 'reddit', True)
[[ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [

### Define Agent Class

In [ ]:
class Agent:
    def __init__(self, matrix_shape, decay=0.9, source_preferences=None):
        self.matrix_shape = matrix_shape
        self.decay = decay

        # Episodic memory by message key.
        self.memory = {}

        # Count-based occurrence history: stores activation ticks only.
        self.occurrences = [
            [[] for _ in range(matrix_shape[1])]
            for _ in range(matrix_shape[0])
        ]

        # Sentiment-weighted history: stores (tick, polarity_value).
        self.occurrences_weighted = [
            [[] for _ in range(matrix_shape[1])]
            for _ in range(matrix_shape[0])
        ]

        if source_preferences is None:
            source_preferences = {}
        self.source_preferences = source_preferences

    def should_consume(self, information_source):
        p = self.source_preferences.get(information_source, 0)
        return np.random.rand() < p

    def receive_matrix(self, matrix, information_source, misinfo_label, tick):
        if not self.should_consume(information_source):
            return

        key = (tick, information_source, misinfo_label)
        self.memory[key] = matrix

        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                val = matrix[i, j]
                if val != 0:
                    self.occurrences[i][j].append(tick)
                    self.occurrences_weighted[i][j].append((tick, float(val)))

    def compute_aggregate_count(self, current_tick):
        agg_matrix = np.zeros(self.matrix_shape)

        for i in range(self.matrix_shape[0]):
            for j in range(self.matrix_shape[1]):
                occ = self.occurrences[i][j]
                if not occ:
                    continue

                t = np.array(occ, dtype=float)
                elapsed = np.maximum(1, current_tick - t)
                decay_values = elapsed ** (-self.decay)
                agg_matrix[i, j] = np.log1p(decay_values.sum())

        return agg_matrix

    def compute_aggregate_weighted(self, current_tick):
        agg_matrix = np.zeros(self.matrix_shape)

        for i in range(self.matrix_shape[0]):
            for j in range(self.matrix_shape[1]):
                occ = self.occurrences_weighted[i][j]
                if not occ:
                    continue

                ticks = np.array([x[0] for x in occ], dtype=float)
                values = np.array([x[1] for x in occ], dtype=float)
                elapsed = np.maximum(1, current_tick - ticks)
                weighted_sum = np.sum(values * (elapsed ** (-self.decay)))

                # Signed log compression keeps sentiment direction while avoiding scale blow-up.
                agg_matrix[i, j] = np.sign(weighted_sum) * np.log1p(np.abs(weighted_sum))

        return agg_matrix

    # Backward-compatible alias for existing plotting code.
    def compute_aggregate(self, current_tick):
        return self.compute_aggregate_count(current_tick)


In [ ]:
# We create a Directory for frames to store the plots that we later create using the plot_matrix function.
# And what happens here is that a folder called 'frames/' is created in our working directory.
frames_dir = "frames"
os.makedirs(frames_dir, exist_ok=True)

### Calibration Agent ...
#### is used to consume all messages and identify the global maximum value across time and across cells of the N-EIM to ensure consistent scaling of plots across simulation runs

In [ ]:
%%time

calibration_agent = Agent(
    matrix_shape=eim_matrix_shape,
    source_preferences={"reddit":1.0, "news":1.0, "gov":1.0}
)

global_max = 0
aggregate_eim_by_tick_count_all_sources = {}
aggregate_eim_by_tick_weighted_all_sources = {}

for (tick, information_source, misinfo_label), matrix in sorted(
        message_passing_dictionary.items(),
        key=lambda item: item[0][0]
    ):

    calibration_agent.receive_matrix(
        matrix,
        information_source,
        misinfo_label,
        tick
    )

    agg_count = calibration_agent.compute_aggregate_count(tick)
    agg_weighted = calibration_agent.compute_aggregate_weighted(tick)

    aggregate_eim_by_tick_count_all_sources[tick] = agg_count.copy()
    aggregate_eim_by_tick_weighted_all_sources[tick] = agg_weighted.copy()

    global_max = max(global_max, agg_count.max())

print("GLOBAL MAX (count track):", global_max)
print("Count-track aggregate matrices:", len(aggregate_eim_by_tick_count_all_sources))
print("Weighted-track aggregate matrices:", len(aggregate_eim_by_tick_weighted_all_sources))


GLOBAL MAX: 2.0610385577735597
Wall time: 1.25 s


### The Final 2 Cells of the Notebook are used to:
#### </t> (1) instantiate agents, </t>
#### </t> (2) define information source preferences,
#### </t> (3) expose agents to messages, and
#### </t> (4) plot visualizations of the N-EIMs imprinted onto agents' cognition over the course of the simulation

In [ ]:
%%time

agent = Agent(
    matrix_shape=eim_matrix_shape,
    source_preferences = {
        "reddit": 0.0,
        "news": 0.0,
        "gov": 1.0
    })

frames = []

for (tick, information_source, misinfo_label), matrix in sorted(
        message_passing_dictionary.items(),
        key=lambda item: item[0][0]
    ):

    agent.receive_matrix(
        matrix=matrix,
        information_source=information_source,
        misinfo_label=misinfo_label,
        tick=tick
    )

    agg = agent.compute_aggregate(current_tick=tick)

    frames.append(agg.copy())


Wall time: 139 ms


In [ ]:
%%time

gif_path = "aggregate_evolution_exclusive_gov source_preference.gif"

with imageio.get_writer(gif_path, mode="I", duration=0.6) as writer:
    for tick, matrix in enumerate(frames, start=1):

        fig, ax = plt.subplots(figsize=(6,6))

        cmap = plt.cm.viridis
        norm = colors.Normalize(vmin=0, vmax=global_max)

        im = ax.imshow(matrix, cmap=cmap, norm=norm, origin='upper')
        fig.colorbar(im, ax=ax)

        ax.set_title(f"Tick {tick}")

        fig.canvas.draw()

        image = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))

        writer.append_data(image)

        plt.close(fig)

Wall time: 4min 38s


In [ ]:
%%time

# Step 4: PCA on aggregate EIM after each tick -> 1-D in [-1, 1] (dual track export)
def run_pca_and_export(aggregate_dict, track_name, out_dir):
    ticks = sorted(aggregate_dict.keys())
    if not ticks:
        return {}, None

    X = np.array([aggregate_dict[t].reshape(-1) for t in ticks], dtype=float)
    X_centered = X - X.mean(axis=0, keepdims=True)
    _, s, vt = np.linalg.svd(X_centered, full_matrices=False)
    pc1_scores = X_centered @ vt[0].T

    # Deterministic sign alignment for reproducibility across runs.
    total_signal = X.sum(axis=1)
    if np.std(total_signal) > 1e-12:
        corr = np.corrcoef(pc1_scores, total_signal)[0, 1]
        if np.isfinite(corr) and corr < 0:
            pc1_scores = -pc1_scores

    pc1_min = pc1_scores.min()
    pc1_max = pc1_scores.max()
    if np.isclose(pc1_min, pc1_max):
        pc1_scaled = np.zeros_like(pc1_scores)
    else:
        pc1_scaled = 2 * (pc1_scores - pc1_min) / (pc1_max - pc1_min) - 1

    eim_1d_by_tick = {int(t): float(v) for t, v in zip(ticks, pc1_scaled)}

    explained_variance = (s ** 2) / max(1, (X_centered.shape[0] - 1))
    explained_ratio_pc1 = float(explained_variance[0] / explained_variance.sum())

    json_path = f"{out_dir}/eim_1d_by_tick_v8_{track_name}_general_topic_general_narrative.json"
    csv_path = f"{out_dir}/eim_1d_by_tick_v8_{track_name}_general_topic_general_narrative.csv"
    meta_path = f"{out_dir}/pca_metadata_v8_{track_name}_general_topic_general_narrative.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(eim_1d_by_tick, f, indent=2)

    pd.DataFrame({"tick": ticks, "eim_1d": pc1_scaled}).to_csv(csv_path, index=False)

    metadata = {
        "track": track_name,
        "matrix_shape_topic_x_narrative": [int(eim_matrix_shape[0]), int(eim_matrix_shape[1])],
        "topic_count": int(len(topic_order)),
        "narrative_count": int(len(narrative_order)),
        "tick_count": int(len(ticks)),
        "pc1_explained_variance_ratio": explained_ratio_pc1,
        "pc1_scaled_min": float(np.min(pc1_scaled)),
        "pc1_scaled_max": float(np.max(pc1_scaled)),
        "pc1_scaled_mean": float(np.mean(pc1_scaled)),
        "pc1_scaled_std": float(np.std(pc1_scaled))
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"[{track_name}] Saved:", json_path)
    print(f"[{track_name}] Saved:", csv_path)
    print(f"[{track_name}] Saved:", meta_path)
    print(f"[{track_name}] PC1 explained variance ratio:", round(explained_ratio_pc1, 4))

    return eim_1d_by_tick, metadata

os.makedirs("pca_outputs_v8", exist_ok=True)

eim_1d_by_tick_count, meta_count = run_pca_and_export(
    aggregate_eim_by_tick_count_all_sources,
    "count",
    "pca_outputs_v8"
)

eim_1d_by_tick_weighted, meta_weighted = run_pca_and_export(
    aggregate_eim_by_tick_weighted_all_sources,
    "weighted",
    "pca_outputs_v8"
)

# Keep compatibility with downstream notebooks expecting one default dictionary.
eim_1d_by_tick = eim_1d_by_tick_weighted

comparison_path = "pca_outputs_v8/pca_dual_track_comparison_v8_general_topic_general_narrative.json"
comparison = {
    "recommended_default_track": "weighted",
    "count_track": meta_count,
    "weighted_track": meta_weighted
}
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)

print("Saved:", comparison_path)
print("Default dictionary variable points to weighted track: eim_1d_by_tick")

print("First 10 weighted entries:")
for t in sorted(eim_1d_by_tick_weighted.keys())[:10]:
    print(t, eim_1d_by_tick_weighted[t])

eim_1d_by_tick_weighted
